In [ ]:
# 대회 환경 맞추기
!pip uninstall -y torch transformers -q

!pip install torch
!pip install transformers==4.57.3
!pip install tokenizers==0.22.1
!pip install accelerate==1.10.1
!pip install datasets==4.4.1
!pip install compressed-tensors==0.13.0

# llmcompressor 설치 시도
!pip install llmcompressor

print("설치 완료 - 런타임 재시작 필요!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 1.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.18.1 requires transformers, which is not installed.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, which is not installed.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 132.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 112.1 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
   

# import

In [2]:
import os
import random
import shutil
from pathlib import Path

import torch
from datasets import load_dataset, Dataset, concatenate_datasets

from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

# 토크나이저 fork 경고(데드락 방지)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Variable definition

In [3]:
MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"
OUT_DIR  = "./model" # 결과 저장 폴더

# Calibration dataset mix
MANTA_ID = "LGAI-EXAONE/MANTA-1M"
MANTA_SPLIT = "train"

GSM8K_ID = "gsm8k"
GSM8K_CONFIG = "main"
GSM8K_SPLIT = "train"

# Calibration
NUM_CALIBRATION_SAMPLES = 1024    # 512
MAX_SEQUENCE_LENGTH = 1024        # 512보다 일반적으로 안정적(단, 속도/메모리 비용 증가)

# Quantization schemes
SCHEME_MAIN = "W4A16"  # 대부분 레이어: GPTQ int4
SCHEME_HI   = "W8A16"  # 민감/에러 집중 레이어: int8

# vLLM 제출 호환을 위해 보통 lm_head는 fp16/bf16 유지 권장
# embed_tokens도 보통 보호 추천 (단, 규칙상 금지는 아님)
PROTECT_ALWAYS = ["lm_head", "embed_tokens"]

# 레이어 보호: "마지막 2개 레이어"
# EXAONE-4.0-1.2B가 30 layers라는 전제(일반적으로 0~29)
PROTECT_LAST_N_LAYERS = 2

# Dataset Load

In [4]:
# =========================================
# [블록 2] Dataset load + text normalize
# =========================================
def pick_text(example: dict) -> str:
    """
    서로 다른 데이터셋에서 텍스트 컬럼이 다를 수 있으니 최대한 안전하게 뽑는다.
    - MANTA: 보통 instruction/input/output 또는 messages 형태일 수 있음
    - GSM8K: question, answer
    """
    # 1) 가장 흔한 단일 텍스트 컬럼들
    for k in ["text", "content", "prompt"]:
        if k in example and isinstance(example[k], str) and example[k].strip():
            return example[k].strip()

    # 2) instruction style
    if "instruction" in example:
        ins = str(example.get("instruction", "")).strip()
        inp = str(example.get("input", "")).strip()
        out = str(example.get("output", "")).strip()
        s = ins
        if inp:
            s += "\n\n" + inp
        if out:
            s += "\n\n" + out
        return s.strip()

    # 3) chat/messages style
    if "messages" in example and isinstance(example["messages"], (list, tuple)):
        # 가능한 한 role/content를 직렬화
        parts = []
        for m in example["messages"]:
            if not isinstance(m, dict):
                continue
            role = str(m.get("role", "")).strip()
            content = str(m.get("content", "")).strip()
            if content:
                parts.append(f"[{role}] {content}" if role else content)
        if parts:
            return "\n".join(parts).strip()

    # 4) GSM8K
    if "question" in example:
        q = str(example.get("question", "")).strip()
        a = str(example.get("answer", "")).strip()
        if q and a:
            return f"Q: {q}\nA: {a}"
        if q:
            return f"Q: {q}"

    # 마지막 fallback: 문자열화
    return str(example)

# Calibration

In [7]:
# =========================================
# [블록 3] Build calibration dataset (80/20 mix)
# =========================================
SEED = 42
random.seed(SEED)

total = NUM_CALIBRATION_SAMPLES
gsm_n = int(total * 0.20)
manta_n = total - gsm_n

print(f"[INFO] calib samples: total={total}, manta={manta_n}, gsm8k={gsm_n}")

print("[INFO] Loading MANTA...")
manta = load_dataset(MANTA_ID, split=MANTA_SPLIT)

print("[INFO] Loading GSM8K...")
gsm8k = load_dataset(GSM8K_ID, GSM8K_CONFIG, split=GSM8K_SPLIT)

# 셔플 후 필요한 개수만 선택
manta = manta.shuffle(seed=SEED).select(range(min(manta_n, len(manta))))
gsm8k = gsm8k.shuffle(seed=SEED).select(range(min(gsm_n, len(gsm8k))))

# 텍스트 컬럼 통일
manta = manta.map(lambda ex: {"text": pick_text(ex)}, remove_columns=manta.column_names)
gsm8k = gsm8k.map(lambda ex: {"text": pick_text(ex)}, remove_columns=gsm8k.column_names)

# (추가) 캘리브레이션 길이 필터링 함수
def filter_length(example):
    """너무 짧거나 긴 샘플 제거하는 용도"""
    tokens = tokenizer(example["text"], truncation=False, add_special_tokens=False)
    token_len = len(tokens["input_ids"])
    # 최소 50 토큰, 최대 MAX_SEQUENCE_LENGTH 토큰
    return 50 <= token_len <= MAX_SEQUENCE_LENGTH

print("[INFO] Filtering by length...")
manta_before = len(manta)
gsm8k_before = len(gsm8k)

manta = manta.filter(filter_length)
gsm8k = gsm8k.filter(filter_length)

print(f"[INFO] MANTA: {manta_before} -> {len(manta)} samples")
print(f"[INFO] GSM8K: {gsm8k_before} -> {len(gsm8k)} samples")

calib = concatenate_datasets([manta, gsm8k]).shuffle(seed=SEED)

print("[INFO] calib dataset ready:", calib)
print("[INFO] example:\n", calib[0]["text"][:300])

[INFO] calib samples: total=1024, manta=820, gsm8k=204
[INFO] Loading MANTA...
[INFO] Loading GSM8K...
[INFO] Filtering by length...


Filter:   0%|          | 0/820 [00:00<?, ? examples/s]

Filter:   0%|          | 0/204 [00:00<?, ? examples/s]

[INFO] MANTA: 820 -> 630 samples
[INFO] GSM8K: 204 -> 204 samples
[INFO] calib dataset ready: Dataset({
    features: ['text'],
    num_rows: 834
})
[INFO] example:
 {'id': 'c9dfe25b50cb2b4f24ca0fb9694ea77f', 'conversations': [{'content': 'Using data from a recent survey of young Japanese adults (ages 18-35) regarding their engagement with traditional practices and dietary habits, analyze how modern lifestyles have influenced the preservation or transformation o


# Load

In [8]:
# =========================================
# [블록 4] Load tokenizer/model
# =========================================
print("[INFO] Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

print("[INFO] Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,     # 제출 서버(vLLM)도 float16 기반이므로 여기서 맞추는 게 안전
    trust_remote_code=True,
)

model.eval()
print("[INFO] Model/tokenizer loaded")

[INFO] Loading tokenizer...
[INFO] Loading model...
[INFO] Model/tokenizer loaded


# Recipe

In [9]:
# =========================================
# [블록 5] Build recipe (mixed precision)
# - o_proj, down_proj : W8A16
# - others Linear     : W4A16
# - last 2 layers protected (fp16)
# =========================================
# 마지막 레이어 인덱스 계산(모델 레이어 수가 다를 수 있으니 실제 len으로 계산)
num_layers = len(model.model.layers)
last_ids = list(range(num_layers - PROTECT_LAST_N_LAYERS, num_layers))
last_re = "|".join(str(i) for i in last_ids)
IGNORE_LAST_LAYERS_REGEX = f"re:^model\\.layers\\.({last_re})\\."

print(f"[INFO] detected num_layers={num_layers}, protect(last {PROTECT_LAST_N_LAYERS})={last_ids}")
print(f"[INFO] ignore regex = {IGNORE_LAST_LAYERS_REGEX}")

# (1) 고비트(W8A16) 타겟: self_attn.o_proj + mlp.down_proj
# regex 타겟을 쓰면 레이어별로 자동 매칭됨
HI_TARGETS = [
    r"re:.*self_attn\.o_proj$",
    r"re:.*mlp\.down_proj$",
]

# (2) 저비트(W4A16) 타겟: 모든 Linear
MAIN_TARGETS = ["Linear"]

# ignore 리스트: 항상 보호 + 마지막 2개 레이어 전부 보호 + (W8으로 갈 애들은 W4에서 제외)
IGNORE_FOR_MAIN = [
    *PROTECT_ALWAYS,
    IGNORE_LAST_LAYERS_REGEX,
    r"re:.*self_attn\.o_proj$",
    r"re:.*mlp\.down_proj$",
]

# ignore 리스트: W8 대상에서도 마지막 2개 레이어는 보호
IGNORE_FOR_HI = [
    *PROTECT_ALWAYS,          # 원하면 여기서 embed/lm_head 빼도 되지만 보통 유지 권장
    IGNORE_LAST_LAYERS_REGEX,
]

recipe = [
    # 먼저 W8A16 (민감/에러 집중 모듈)
    # (추가) actorder, dampening_frac = 0.1
    GPTQModifier(
        scheme=SCHEME_HI,
        targets=HI_TARGETS,
        ignore=IGNORE_FOR_HI,
        actorder="static", # (추가)
        dampening_frac=0.1, # (추가)
    ),
    # 그 다음 W4A16 (나머지 Linear)
    GPTQModifier(
        scheme=SCHEME_MAIN,
        targets=MAIN_TARGETS,
        ignore=IGNORE_FOR_MAIN,
        actorder="static", # (추가)
        dampening_frac=0.1, # (추가)
    ),
]

print("[INFO] recipe ready")

[INFO] detected num_layers=30, protect(last 2)=[28, 29]
[INFO] ignore regex = re:^model\.layers\.(28|29)\.
[INFO] recipe ready


# Run / Save

In [10]:
# =========================================
# [블록 6] Run oneshot GPTQ and save
# =========================================
os.makedirs(OUT_DIR, exist_ok=True)

print("[INFO] Running oneshot GPTQ... (this may take a while)")
oneshot(
    model=model,
    tokenizer=tokenizer,
    dataset=calib,                      # 80/20 혼합 데이터
    num_calibration_samples=total,      # 이미 calib를 total로 만들었지만, 명시적으로 동일 값
    max_seq_length=MAX_SEQUENCE_LENGTH,
    output_dir=OUT_DIR,
    recipe=recipe,
)

print(f"[DONE] Saved to: {OUT_DIR}")

[INFO] Running oneshot GPTQ... (this may take a while)


Tokenizing:   0%|          | 0/834 [00:00<?, ? examples/s]

2026-02-23T12:26:43.733598+0000 | _make_sampler | WARNING - Requested 1024 samples but the provided dataset only has 834 samples.
2026-02-23T12:26:44.084241+0000 | reset | INFO - Compression lifecycle reset
2026-02-23T12:26:44.086674+0000 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-23T12:26:44.138782+0000 | initialize | INFO - Compression lifecycle initialized for 2 modifiers
2026-02-23T12:26:44.139604+0000 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`


(1/31): Calibrating: 100%|██████████| 834/834 [00:09<00:00, 83.41it/s]

2026-02-23T12:26:55.709244+0000 | compress_modules | INFO - Quantizing model.layers.0.self_attn.o_proj using 834 samples


...

(31/31): Propagating: 100%|██████████| 834/834 [00:01<00:00, 707.00it/s]

2026-02-23T12:43:48.496566+0000 | finalize | INFO - Compression lifecycle finalized for 2 modifiers


2026-02-23T12:43:48.555339+0000 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 196it [00:02, 74.77it/s]


[DONE] Saved to: ./model


#Submission

In [11]:
zip_name = "GPTQ_0.61"
print(f"[INFO] {zip_name}.zip 생성 중...")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

print(f"[INFO] 생성 완료: {zip_name}.zip")

[INFO] GPTQ_0.61.zip 생성 중...
[INFO] 생성 완료: GPTQ_0.61.zip
